# Netflix Content Analytics — EDA & Information Extraction
**Person B (Manpreet Kaur — 33647884)**

This notebook performs:
1. Exploratory Data Analysis (summary stats, missingness, distributions, relationships)
2. Information Extraction (genre trends, country shifts, movies vs TV shows)
3. Exports visualizations for the report

---
## Setup & Data Loading
Upload `netflix_titles_clean.csv` using the file upload cell below.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
from google.colab import files

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150

# Upload the cleaned CSV
uploaded = files.upload()  

# Load data
DATA_PATH = "netflix_titles_clean.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["date_added"])
print(f"Dataset shape: {df.shape}")
df.head()

---
## 1. Summary Statistics

In [ ]:
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

print(f"\nTotal titles: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Date range of additions: {df['date_added'].min().date()} to {df['date_added'].max().date()}")
print(f"Release year range: {df['release_year'].min()} to {df['release_year'].max()}")

print("\n--- Content Type Breakdown ---")
print(df["type"].value_counts())
print(f"\nMovie-to-TV-Show ratio: {df['type'].value_counts()['Movie'] / df['type'].value_counts()['TV Show']:.2f}:1")

print("\n--- Numeric Column Statistics ---")
df[["release_year", "year_added", "month_added", "duration_value"]].describe().round(1)

---
## 2. Missingness Analysis

Even after cleaning (where Unknown was used for missing director/cast/country),
we examine "Unknown" rates to understand data limitations.

In [ ]:
print("=" * 60)
print("MISSINGNESS ANALYSIS (post-cleaning 'Unknown' rates)")
print("=" * 60)

unknown_cols = ["director", "cast", "country"]
miss_data = []
for col in unknown_cols:
    n_unknown = (df[col] == "Unknown").sum()
    pct = n_unknown / len(df) * 100
    miss_data.append({"Column": col, "Unknown Count": n_unknown, "% Unknown": f"{pct:.1f}%"})
    print(f"  {col}: {n_unknown:,} Unknown ({pct:.1f}%)")

miss_df = pd.DataFrame(miss_data)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(miss_df["Column"], miss_df["Unknown Count"],
               color=["#E50914", "#B81D24", "#831010"])
ax.set_xlabel("Number of 'Unknown' Values")
ax.set_title("Missing Data After Cleaning (Filled as 'Unknown')")
for bar, pct in zip(bars, miss_df["% Unknown"]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            pct, va="center", fontsize=10)
plt.tight_layout()
plt.show()

---
## 3. Distribution Plots

### 3a. Titles Added Per Year

In [ ]:
yearly = df.groupby("year_added").size().reset_index(name="count")

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly["year_added"], yearly["count"], color="#E50914", edgecolor="white")
ax.set_xlabel("Year Added to Netflix")
ax.set_ylabel("Number of Titles")
ax.set_title("Netflix Content Growth: Titles Added Per Year")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for i, row in yearly.iterrows():
    if row["count"] > 200:
        ax.text(row["year_added"], row["count"] + 20, str(row["count"]),
                ha="center", fontsize=8)
plt.tight_layout()
plt.show()

### 3b. Genre Frequency (Top 15)

In [ ]:
genre_counts = df["primary_genre"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
genre_counts.plot(kind="barh", ax=ax, color="#E50914")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("Primary Genre")
ax.set_title("Top 15 Genres on Netflix")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 3c. Country Frequency (Top 15)

In [ ]:
country_counts = df[df["primary_country"] != "Unknown"]["primary_country"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
country_counts.plot(kind="barh", ax=ax, color="#221F1F")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("Country")
ax.set_title("Top 15 Content-Producing Countries on Netflix")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 3d. Content Rating Distribution

In [ ]:
rating_counts = df["rating"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
rating_counts.plot(kind="bar", ax=ax, color="#B81D24", edgecolor="white")
ax.set_xlabel("Rating")
ax.set_ylabel("Number of Titles")
ax.set_title("Content Rating Distribution")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

---
## 4. Relationship Analysis

### 4a. Movies vs. TV Shows Over Time

In [ ]:
type_year = df.groupby(["year_added", "type"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
type_year.plot(kind="bar", stacked=True, ax=ax, color=["#E50914", "#221F1F"],
               edgecolor="white", width=0.8)
ax.set_xlabel("Year Added to Netflix")
ax.set_ylabel("Number of Titles")
ax.set_title("Movies vs. TV Shows Added Per Year")
ax.legend(title="Type")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Compute TV show share over time
tv_share = (type_year["TV Show"] / type_year.sum(axis=1) * 100).round(1)
print("\nTV Show share of new additions (%):")
print(tv_share.tail(8).to_string())

### 4b. Genre vs. Year — Heatmap of Top 8 Genres Over Time

In [ ]:
top8_genres = df["primary_genre"].value_counts().head(8).index.tolist()
genre_year = (df[df["primary_genre"].isin(top8_genres)]
              .groupby(["year_added", "primary_genre"])
              .size()
              .unstack(fill_value=0))

# Filter to years with meaningful data
genre_year = genre_year[genre_year.index >= 2015]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(genre_year.T, cmap="Reds", annot=True, fmt="d", linewidths=0.5,
            ax=ax, cbar_kws={"label": "Title Count"})
ax.set_xlabel("Year Added")
ax.set_ylabel("Genre")
ax.set_title("Genre Popularity by Year (Top 8 Genres, 2015–2021)")
plt.tight_layout()
plt.show()

### 4c. Country vs. Content Type

In [ ]:
top10_countries = df[df["primary_country"] != "Unknown"]["primary_country"].value_counts().head(10).index.tolist()
country_type = (df[df["primary_country"].isin(top10_countries)]
                .groupby(["primary_country", "type"])
                .size()
                .unstack(fill_value=0))
country_type = country_type.sort_values("Movie", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
country_type.plot(kind="barh", stacked=True, ax=ax, color=["#E50914", "#221F1F"])
ax.set_xlabel("Number of Titles")
ax.set_ylabel("Country")
ax.set_title("Content Type Mix by Top 10 Producing Countries")
ax.legend(title="Type")
plt.tight_layout()
plt.show()

---
## ⭐ KEY EDA INSIGHT → Shapes Information Extraction

**Finding: International content (especially from India) surged dramatically after 2016,
and Documentaries/Stand-Up Comedy grew as distinct genres during 2017–2019.**

This insight directly shapes our information extraction:
- We will **focus the genre trend analysis on the post-2015 era** where changes are most dramatic
- We will **track India's rise as a content contributor** year-over-year
- We will examine whether the **TV Show share increase** correlates with specific genres or countries

These findings were not obvious before the EDA — they emerged from the heatmap
and the country-over-time analysis above.

---
## 5. Information Extraction

Guided by the EDA insight above, we now extract structured findings.

### 5a. Genre Popularity Trends (Year × Genre)

In [ ]:
print("=" * 60)
print("INFORMATION EXTRACTION: Genre Popularity Trends")
print("=" * 60)

# All genres over time
genre_year_full = (df.groupby(["year_added", "primary_genre"])
                   .size()
                   .reset_index(name="count"))

# Focus on post-2015 (EDA-driven decision)
genre_post2015 = genre_year_full[genre_year_full["year_added"] >= 2015]

# Top 6 genres for trend analysis
top6 = df[df["year_added"] >= 2015]["primary_genre"].value_counts().head(6).index.tolist()
genre_trends = genre_post2015[genre_post2015["primary_genre"].isin(top6)]

fig, ax = plt.subplots(figsize=(12, 6))
for genre in top6:
    subset = genre_trends[genre_trends["primary_genre"] == genre]
    ax.plot(subset["year_added"], subset["count"], marker="o", linewidth=2, label=genre)
ax.set_xlabel("Year Added to Netflix")
ax.set_ylabel("Number of Titles")
ax.set_title("Genre Popularity Trends (Top 6 Genres, 2015–2021)\n[Focused post-2015 per EDA insight]")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

# Print key insight
print("\n--- Key Genre Insights ---")
for genre in top6:
    sub = genre_trends[genre_trends["primary_genre"] == genre].sort_values("year_added")
    if len(sub) >= 2:
        first_val = sub.iloc[0]["count"]
        last_val = sub.iloc[-1]["count"]
        change = ((last_val - first_val) / first_val * 100) if first_val > 0 else 0
        peak_year = sub.loc[sub["count"].idxmax(), "year_added"]
        print(f"  {genre}: {first_val} → {last_val} titles ({change:+.0f}%), peaked in {peak_year}")

### 5b. Production Country Shifts (Year × Country)

In [ ]:
print("=" * 60)
print("INFORMATION EXTRACTION: Production Country Shifts")
print("=" * 60)

top5_countries = ["United States", "India", "United Kingdom", "Japan", "South Korea"]
country_year = (df[df["primary_country"].isin(top5_countries)]
                .groupby(["year_added", "primary_country"])
                .size()
                .reset_index(name="count"))

fig, ax = plt.subplots(figsize=(12, 6))
for country in top5_countries:
    subset = country_year[country_year["primary_country"] == country]
    ax.plot(subset["year_added"], subset["count"], marker="o", linewidth=2, label=country)
ax.set_xlabel("Year Added to Netflix")
ax.set_ylabel("Number of Titles")
ax.set_title("Content Production by Top 5 Countries Over Time")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

# India's Rise
india = country_year[country_year["primary_country"] == "India"].sort_values("year_added")
us = country_year[country_year["primary_country"] == "United States"].sort_values("year_added")

print("\n--- India's Rise as Content Contributor ---")
for _, row in india.iterrows():
    us_val = us[us["year_added"] == row["year_added"]]["count"].values
    us_count = us_val[0] if len(us_val) > 0 else 0
    pct_of_us = (row["count"] / us_count * 100) if us_count > 0 else 0
    print(f"  {int(row['year_added'])}: India added {int(row['count'])} titles ({pct_of_us:.0f}% of US output)")

# Country rank by year
print("\n--- Top Content Producer Rank by Year (2017–2021) ---")
for year in range(2017, 2022):
    yr_data = (df[df["year_added"] == year]
               .query("primary_country != 'Unknown'")
               ["primary_country"].value_counts().head(3))
    top3 = ", ".join([f"{c} ({n})" for c, n in yr_data.items()])
    print(f"  {year}: {top3}")

### 5c. Movies vs. TV Shows — Growth Rate Comparison

In [ ]:
print("=" * 60)
print("INFORMATION EXTRACTION: Movies vs. TV Shows Growth")
print("=" * 60)

type_by_year = df.groupby(["year_added", "type"]).size().unstack(fill_value=0)
type_by_year["Total"] = type_by_year.sum(axis=1)
type_by_year["TV_Share_%"] = (type_by_year["TV Show"] / type_by_year["Total"] * 100).round(1)
type_by_year["Movie_Share_%"] = (type_by_year["Movie"] / type_by_year["Total"] * 100).round(1)

# Growth rates
type_by_year["Movie_Growth_%"] = type_by_year["Movie"].pct_change().mul(100).round(1)
type_by_year["TVShow_Growth_%"] = type_by_year["TV Show"].pct_change().mul(100).round(1)

print("\nYear-by-Year Breakdown:")
display_cols = ["Movie", "TV Show", "Total", "TV_Share_%", "Movie_Growth_%", "TVShow_Growth_%"]
print(type_by_year[display_cols].tail(8).to_string())

# TV share trend chart
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(type_by_year.index, type_by_year["Movie"], label="Movies", color="#E50914", alpha=0.7)
ax1.bar(type_by_year.index, type_by_year["TV Show"], bottom=type_by_year["Movie"],
        label="TV Shows", color="#221F1F", alpha=0.7)
ax1.set_xlabel("Year Added")
ax1.set_ylabel("Number of Titles")
ax1.set_title("Movies vs. TV Shows: Volume & TV Share Trend")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(type_by_year.index, type_by_year["TV_Share_%"], color="#FFD700",
         marker="D", linewidth=2, label="TV Show Share %")
ax2.set_ylabel("TV Show Share (%)", color="#FFD700")
ax2.legend(loc="upper right")
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

# Key insight
recent = type_by_year[type_by_year.index >= 2017]
avg_tv_share = recent["TV_Share_%"].mean()
print(f"\n📊 Named Insight: TV Show share averaged {avg_tv_share:.1f}% of new additions from 2017 onward,")
print(f"   up from ~15% before 2015 — Netflix has steadily increased TV Show investment.")

### 5d. Duration Analysis — Movies & TV Show Seasons

In [ ]:
print("=" * 60)
print("INFORMATION EXTRACTION: Duration Trends")
print("=" * 60)

# Movie duration over time
movies = df[df["type"] == "Movie"]
movie_dur = movies.groupby("year_added")["duration_value"].agg(["mean", "median", "count"]).round(1)
movie_dur.columns = ["avg_minutes", "median_minutes", "n_movies"]

print("\nAverage Movie Duration by Year Added:")
print(movie_dur[movie_dur.index >= 2015].to_string())

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(movie_dur.index, movie_dur["avg_minutes"], marker="o", color="#E50914",
        linewidth=2, label="Mean Duration")
ax.plot(movie_dur.index, movie_dur["median_minutes"], marker="s", color="#221F1F",
        linewidth=2, linestyle="--", label="Median Duration")
ax.set_xlabel("Year Added to Netflix")
ax.set_ylabel("Duration (minutes)")
ax.set_title("Average Movie Duration Over Time")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# TV show seasons
shows = df[df["type"] == "TV Show"]
season_dist = shows["duration_value"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
season_dist.head(10).plot(kind="bar", ax=ax, color="#E50914", edgecolor="white")
ax.set_xlabel("Number of Seasons")
ax.set_ylabel("Number of TV Shows")
ax.set_title("Distribution of TV Show Season Counts")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

single_season_pct = (shows["duration_value"] == 1).sum() / len(shows) * 100
print(f"\n📊 Named Insight: {single_season_pct:.1f}% of TV shows on Netflix have only 1 season.")

---
## 6. Summary of Named Insights

| # | Insight | Evidence |
|---|---------|----------|
| 1 | **Netflix content additions peaked around 2019 before declining** | Titles-per-year bar chart shows rapid growth 2015–2019, then decline |
| 2 | **International Movies and Dramas dominate the genre landscape** | Top genres chart: International Movies and Dramas each have 2,000+ titles |
| 3 | **India became the #2 content contributor** | Country trends: India overtook UK/Canada by 2018–2019 |
| 4 | **TV Shows' share of new content has been rising** | TV share grew from ~15% pre-2015 to ~30%+ by 2019–2021 |
| 5 | **Documentaries and Stand-Up surged post-2016** | Genre heatmap shows clear growth in these categories |
| 6 | **~70%+ of TV shows have only 1 season** | Season distribution chart — limited series dominate |
| 7 | **Average movie duration has remained stable (~90–100 min)** | Duration trend line is relatively flat |

**Insight → Extraction Connection:** Insight #3 (India's rise) and #5 (Documentary/Stand-Up surge)
directly motivated focusing the genre and country extraction on the **post-2015 period**,
where changes are most visible and analytically interesting.

---
## 7. Export Summary Statistics for Report

In [ ]:
print("=" * 60)
print("SUMMARY FOR REPORT")
print("=" * 60)

total = len(df)
movies_n = (df["type"] == "Movie").sum()
shows_n = (df["type"] == "TV Show").sum()
n_countries = df[df["primary_country"] != "Unknown"]["primary_country"].nunique()
n_genres = df["primary_genre"].nunique()

print(f"""
📋 DATASET SUMMARY
   Total titles: {total:,}
   Movies: {movies_n:,} ({movies_n/total*100:.1f}%)
   TV Shows: {shows_n:,} ({shows_n/total*100:.1f}%)
   Unique countries: {n_countries}
   Unique primary genres: {n_genres}
   Date range: {df['date_added'].min().date()} to {df['date_added'].max().date()}

📊 KEY NUMBERS FOR REPORT
   Peak year for additions: {yearly.loc[yearly['count'].idxmax(), 'year_added']} ({yearly['count'].max()} titles)
   Top genre: {df['primary_genre'].value_counts().index[0]} ({df['primary_genre'].value_counts().values[0]} titles)
   Top country: {df[df['primary_country']!='Unknown']['primary_country'].value_counts().index[0]}
   Single-season TV shows: {single_season_pct:.1f}%
""")

print("\n✅ All visualizations generated inline above.")
print("📝 Right-click each chart in Colab to save as PNG for your report.")